In [ ]:
import pandas as pd

In [ ]:
path = "/content/task-a-en.tsv"
df = pd.read_csv(path, sep='\t')

print("Shape:", df.shape)
df.head(10)

Shape: (300, 4)


,id,word1,word2,headline
0,en_2001,-,-,Panamanian lawmakers’ Taiwan trip sparks diplo...
1,en_2002,-,-,Carra: Liverpool must sign in January - even a...
2,en_2003,-,-,Experts warn of health threat posed by ultra-p...
3,en_2004,-,-,Dick Cheney funeral: George W. Bush delivers e...
4,en_2005,-,-,No place for Mostert distraction in South Afri...
5,en_2006,-,-,Meet Troy - the parrot named after Ireland’s h...
6,en_2007,-,-,Instagram to start closing Australian teen acc...
7,en_2008,-,-,How the EU botched its attempt to regulate AI
8,en_2009,-,-,I stayed at one of only two hotels in Malta’s ...
9,en_2010,-,-,Ireland coach Farrell brings back Bundee Aki f...


In [ ]:
df.columns

Index(['id', 'word1', 'word2', 'headline'], dtype='object')

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 300 entries, 0 to 299
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   id        300 non-null    object
 1   word1     300 non-null    object
 2   word2     300 non-null    object
 3   headline  300 non-null    object
dtypes: object(4)
memory usage: 9.5+ KB


In [ ]:
df.describe(include="all")

,id,word1,word2,headline
count,1200,1200,1200,1200
unique,1200,22,42,1101
top,en_1184,-,-,-
freq,1,1100,1100,100


In [ ]:
# Convert '-' into proper missing values for easier logic
df_clean = df.replace("-", pd.NA)

word_constraint = df_clean["word1"].notna() & df_clean["word2"].notna()
headline_constraint = df_clean["headline"].notna()

print("Word-inclusion rows:", int(word_constraint.sum()))
print("Headline rows:", int(headline_constraint.sum()))
print("Total:", len(df_clean))

Word-inclusion rows: 100
Headline rows: 1100
Total: 1200


In [ ]:
both = word_constraint & headline_constraint
neither = (~word_constraint) & (~headline_constraint)

print("Both present (should be 0):", int(both.sum()))
print("Neither present (should be 0):", int(neither.sum()))

if both.any():
    display(df_clean[both].head())

if neither.any():
    display(df_clean[neither].head())


Both present (should be 0): 0
Neither present (should be 0): 0


In [ ]:
print("Examples: Word-inclusion constraint")
display(df_clean[word_constraint][["id", "word1", "word2"]].head(5))

Examples: Word-inclusion constraint


,id,word1,word2
100,en_0101,spray,chair
101,en_0102,hammer,banana
102,en_0103,move,fridge
103,en_0104,hammer,pumpkin
104,en_0105,peel,pumpkin


In [ ]:
print("Examples: Headline constraint")
display(df_clean[headline_constraint][["id", "headline"]].head(5))

Examples: Headline constraint


,id,headline
0,en_0001,Ryanair to cut 1 million more passenger seats ...
1,en_0002,"Looted by Nazis, a 17th-Century Painting Resur..."
2,en_0003,Analysis: Spotlight on childcare reforms revea...
3,en_0004,Do body wipes actually work? Experts weigh in
4,en_0005,Is Meghan’s Netflix series another ‘exercise i...


In [ ]:
word_rows = df_clean[word_constraint][["id", "word1", "word2"]].reset_index(drop=True)
print("Total word-inclusion rows:", len(word_rows))
word_rows.head()

Total word-inclusion rows: 100


,id,word1,word2
0,en_0101,spray,chair
1,en_0102,hammer,banana
2,en_0103,move,fridge
3,en_0104,hammer,pumpkin
4,en_0105,peel,pumpkin


In [ ]:
i = 0
row = word_rows.iloc[i]
row

,0
id,en_0101
word1,spray
word2,chair


In [ ]:
# strip whitespace everywhere (important!)
df = df.applymap(lambda x: x.strip() if isinstance(x, str) else x)

# treat "-" as missing
df = df.replace({"-": pd.NA, "–": pd.NA, "—": pd.NA})

/tmp/ipython-input-1894807712.py:2: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: x.strip() if isinstance(x, str) else x)


In [ ]:
print("Head:")
display(df.head(5))

Head:


,id,word1,word2,headline
0,en_0001,<NA>,<NA>,Ryanair to cut 1 million more passenger seats ...
1,en_0002,<NA>,<NA>,"Looted by Nazis, a 17th-Century Painting Resur..."
2,en_0003,<NA>,<NA>,Analysis: Spotlight on childcare reforms revea...
3,en_0004,<NA>,<NA>,Do body wipes actually work? Experts weigh in
4,en_0005,<NA>,<NA>,Is Meghan’s Netflix series another ‘exercise i...


In [ ]:
import requests, re, time
from functools import lru_cache

def norm_term(term: str) -> str:
    term = term.strip().lower()
    term = re.sub(r"\s+", "_", term)
    term = re.sub(r"[^a-z0-9_'-]", "", term)
    return term

@lru_cache(maxsize=5000)
def conceptnet_edges(term: str, limit: int = 50, retries: int = 4, sleep: float = 0.8):
    """
    Robust ConceptNet neighbor fetch:
    - skips invalid terms
    - retries on temporary errors (like 502)
    """
    if term is None:
        return []
    term = str(term).strip()
    if term == "" or term == "-" or term.lower() == "nan":
        return []

    t = norm_term(term)
    if t == "" or t == "-":
        return []

    url = f"https://api.conceptnet.io/c/en/{t}?limit={limit}"

    last_err = None
    for attempt in range(retries):
        try:
            r = requests.get(url, timeout=20)
            if r.status_code >= 500:
                # temporary server error
                last_err = f"HTTP {r.status_code}"
                time.sleep(sleep * (attempt + 1))
                continue
            r.raise_for_status()
            data = r.json()

            edges = []
            for e in data.get("edges", []):
                rel = e["rel"]["label"]
                start = e["start"]["label"]
                end = e["end"]["label"]
                weight = float(e.get("weight", 1.0))
                edges.append((rel, start, end, weight))
            return edges

        except Exception as e:
            last_err = repr(e)
            time.sleep(sleep * (attempt + 1))

    # if we fail, return empty list instead of crashing
    print(f"[ConceptNet warn] failed for term='{term}' url='{url}' err={last_err}")
    return []


In [ ]:
row = word_rows.iloc[0]
w1, w2 = row["word1"], row["word2"]
print("Selected:", row["id"], "|", w1, "|", w2)

edges1 = conceptnet_edges(w1, limit=50)
edges2 = conceptnet_edges(w2, limit=50)

print("Edges fetched:", len(edges1), len(edges2))

Selected: en_0101 | spray | chair
[ConceptNet warn] failed for term='spray' url='https://api.conceptnet.io/c/en/spray?limit=50' err=HTTP 502
[ConceptNet warn] failed for term='chair' url='https://api.conceptnet.io/c/en/chair?limit=50' err=HTTP 502
Edges fetched: 0 0


In [ ]:
def neighbor_set(edges, focus_word, maxn=60):
    focus = str(focus_word).lower()
    neigh = []
    for rel, start, end, weight in edges:
        if start.lower() == focus:
            neigh.append((end, rel, weight))
        elif end.lower() == focus:
            neigh.append((start, rel, weight))
    neigh = sorted(neigh, key=lambda x: x[2], reverse=True)[:maxn]
    return neigh

neigh1 = neighbor_set(edges1, w1, maxn=40)
neigh2 = neighbor_set(edges2, w2, maxn=40)

print("\nTop neighbors for", w1)
for x in neigh1[:10]:
    print(" ", x)

print("\nTop neighbors for", w2)
for x in neigh2[:10]:
    print(" ", x)

overlap = sorted(list(set([n[0].lower() for n in neigh1]).intersection(set([n[0].lower() for n in neigh2]))))
print("\nOverlap bridge candidates:", overlap[:25])
print("Total overlap:", len(overlap))



Top neighbors for spray

Top neighbors for chair

Overlap bridge candidates: []
Total overlap: 0


WordNet

In [ ]:
import nltk
nltk.download("wordnet")
nltk.download("omw-1.4")
from nltk.corpus import wordnet as wn

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


In [ ]:
from collections import defaultdict

def synset_neighbors(syn, max_per_rel=8):
    rels = defaultdict(list)
    rels["hypernyms"] = syn.hypernyms()[:max_per_rel]
    rels["hyponyms"] = syn.hyponyms()[:max_per_rel]
    rels["part_meronyms"] = syn.part_meronyms()[:max_per_rel]
    rels["member_holonyms"] = syn.member_holonyms()[:max_per_rel]
    rels["similar_tos"] = syn.similar_tos()[:max_per_rel]
    return rels

def wordnet_subgraph(word, top_senses=2):
    synsets = wn.synsets(word)[:top_senses]
    out = []
    for syn in synsets:
        out.append({
            "synset": syn.name(),
            "pos": syn.pos(),
            "lemmas": syn.lemma_names(),
            "definition": syn.definition(),
            "examples": syn.examples(),
            "neighbors": {k: [s.name() for s in v] for k, v in synset_neighbors(syn).items()}
        })
    return out


In [ ]:
row = word_rows.iloc[0]
w1, w2 = row["word1"], row["word2"]
print("Selected:", row["id"], w1, w2)

sg1 = wordnet_subgraph(w1, top_senses=2)
sg2 = wordnet_subgraph(w2, top_senses=2)

sg1, sg2

Selected: en_0101 spray chair


([{'synset': 'spray.n.01',
   'pos': 'n',
   'lemmas': ['spray'],
   'definition': 'a pesticide in suspension or solution; intended for spraying',
   'examples': [],
   'neighbors': {'hypernyms': ['pesticide.n.01'],
    'hyponyms': [],
    'part_meronyms': [],
    'member_holonyms': [],
    'similar_tos': []}},
  {'synset': 'spray.n.02',
   'pos': 'n',
   'lemmas': ['spray', 'spraying'],
   'definition': 'a quantity of small objects flying through the air',
   'examples': ['a spray of bullets'],
   'neighbors': {'hypernyms': ['small_indefinite_quantity.n.01'],
    'hyponyms': [],
    'part_meronyms': [],
    'member_holonyms': [],
    'similar_tos': []}}],
 [{'synset': 'chair.n.01',
   'pos': 'n',
   'lemmas': ['chair'],
   'definition': 'a seat for one person, with a support for the back',
   'examples': ['he put his coat over the back of the chair and sat down'],
   'neighbors': {'hypernyms': ['seat.n.03'],
    'hyponyms': ['straight_chair.n.01',
     'ladder-back.n.01',
     'chair_

In [ ]:
def hypernym_chain(syn, depth=6):
    chain = []
    cur = syn
    for _ in range(depth):
        hypers = cur.hypernyms()
        if not hypers:
            break
        cur = hypers[0]
        chain.append(cur)
    return chain

def hypernym_set(word, senses=3, depth=6):
    hs = set()
    for syn in wn.synsets(word)[:senses]:
        hs.add(syn)
        for h in hypernym_chain(syn, depth=depth):
            hs.add(h)
    return hs

def bridge_hypernyms(w1, w2, senses=3, depth=6, topk=10):
    h1 = hypernym_set(w1, senses=senses, depth=depth)
    h2 = hypernym_set(w2, senses=senses, depth=depth)
    overlap = list(h1.intersection(h2))
    # Sort: more specific first (rough heuristic: longer name)
    overlap = sorted(overlap, key=lambda s: len(s.name()), reverse=True)
    return [s.name() for s in overlap[:topk]]


In [ ]:
import re

STOP = set(["the","a","an","and","or","to","of","in","on","for","with","is","are","was","were","be","as","at","by","from","that","this","it","into"])

def gloss_keywords(word, senses=3):
    kws = set()
    for syn in wn.synsets(word)[:senses]:
        words = re.findall(r"[a-zA-Z]+", syn.definition().lower())
        kws.update([w for w in words if w not in STOP and len(w) > 3])
    return kws

def bridge_gloss(w1, w2, senses=3, topk=20):
    k1 = gloss_keywords(w1, senses=senses)
    k2 = gloss_keywords(w2, senses=senses)
    overlap = sorted(list(k1.intersection(k2)))
    return overlap[:topk]


In [ ]:
bh = bridge_hypernyms(w1, w2)
bg = bridge_gloss(w1, w2)

print("Hypernym bridges:", bh)
print("Gloss keyword bridges:", bg)


Hypernym bridges: ['entity.n.01']
Gloss keyword bridges: []


In [ ]:
import random

def guess_pos(word):
    syns = wn.synsets(word)
    return syns[0].pos() if syns else "n"  # default noun

def plan_strategy_wordnet(w1, w2, bh, bg, seed=0):
    rng = random.Random(seed)
    pos1, pos2 = guess_pos(w1), guess_pos(w2)

    if pos1 != pos2:
        return rng.choice(["role_reversal", "literal_misread", "misunderstood_job"])
    # both nouns
    if pos1 == "n" and pos2 == "n":
        if len(bg) > 0:
            return rng.choice(["analogy", "deadpan_equation", "overly_logical"])
        return rng.choice(["anthropomorphism", "product_mashup", "fake_warning_label"])
    # both verbs or other
    return rng.choice(["overly_logical", "exaggeration", "misdirection"])


In [ ]:
def twist_plan_wordnet(w1, w2, bh, bg, seed=0):
    strat = plan_strategy_wordnet(w1, w2, bh, bg, seed=seed)

    # choose an "anchor" from gloss overlap if available, else from hypernym bridge
    anchor = None
    if bg:
        anchor = bg[0]
    elif bh:
        anchor = bh[0]
    else:
        anchor = "unexpected connection"

    # plan templates
    if strat == "anthropomorphism":
        setup = f"Make {w2} act like a person."
        twist = f"Use {w1} as its 'personality' or coping mechanism."
    elif strat == "product_mashup":
        setup = f"Invent a ridiculous product that combines {w1} and {w2}."
        twist = f"Anchor the joke around '{anchor}'."
    elif strat == "fake_warning_label":
        setup = f"Write like an instruction label or safety warning."
        twist = f"The warning reveals why {w1} and {w2} are a terrible combo."
    elif strat == "role_reversal":
        setup = f"Let {w2} do the action to the human."
        twist = f"{w1} becomes the tool/weapon in an unexpected way."
    elif strat == "analogy":
        setup = f"Compare {w1} to {w2} using '{anchor}'."
        twist = "Punchline should flip the comparison at the end."
    else:  # default
        setup = f"Start normal, include both {w1} and {w2} naturally."
        twist = f"Reveal a surprising link via '{anchor}'."

    return {
        "word1": w1,
        "word2": w2,
        "strategy": strat,
        "anchor": anchor,
        "setup_guidance": setup,
        "twist_guidance": twist,
        "must_include": [w1, w2],
    }


In [ ]:
plan = twist_plan_wordnet(w1, w2, bh, bg, seed=42)
plan


{'word1': 'spray',
 'word2': 'chair',
 'strategy': 'fake_warning_label',
 'anchor': 'entity.n.01',
 'setup_guidance': 'Write like an instruction label or safety warning.',
 'twist_guidance': 'The warning reveals why spray and chair are a terrible combo.',
 'must_include': ['spray', 'chair']}

In [ ]:
def build_wordnet_plan_for_row(row, seed=0):
    w1, w2 = row["word1"], row["word2"]
    bh = bridge_hypernyms(w1, w2)
    bg = bridge_gloss(w1, w2)
    plan = twist_plan_wordnet(w1, w2, bh, bg, seed=seed)
    return {
        "id": row["id"],
        "word1": w1,
        "word2": w2,
        "hypernym_bridges": bh[:5],
        "gloss_bridges": bg[:10],
        "plan": plan,
    }

for i in range(3):
    r = word_rows.iloc[i]
    out = build_wordnet_plan_for_row(r, seed=100+i)
    print("\n==", out["id"], "==")
    print("Words:", out["word1"], "|", out["word2"])
    print("Hypernym bridges:", out["hypernym_bridges"])
    print("Gloss bridges:", out["gloss_bridges"])
    print("Strategy:", out["plan"]["strategy"])
    print("Anchor:", out["plan"]["anchor"])
    print("Setup:", out["plan"]["setup_guidance"])
    print("Twist:", out["plan"]["twist_guidance"])



== en_0101 ==
Words: spray | chair
Hypernym bridges: ['entity.n.01']
Gloss bridges: []
Strategy: anthropomorphism
Anchor: entity.n.01
Setup: Make chair act like a person.
Twist: Use spray as its 'personality' or coping mechanism.

== en_0102 ==
Words: hammer | banana
Hypernym bridges: ['whole.n.02']
Gloss bridges: []
Strategy: fake_warning_label
Anchor: whole.n.02
Setup: Write like an instruction label or safety warning.
Twist: The warning reveals why hammer and banana are a terrible combo.

== en_0103 ==
Words: move | fridge
Hypernym bridges: []
Gloss bridges: []
Strategy: anthropomorphism
Anchor: unexpected connection
Setup: Make fridge act like a person.
Twist: Use move as its 'personality' or coping mechanism.


Next Step

In [ ]:
!pip -q install -U transformers accelerate bitsandbytes sentencepiece huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 65.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.6/536.6 kB 28.4 MB/s eta 0:00:00


In [ ]:
# Llama 3.1 (recommended)
# MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"   # :contentReference[oaicite:3]{index=3}

# OR Llama 2 chat
# MODEL_ID = "meta-llama/Llama-2-7b-chat-hf"    # :contentReference[oaicite:4]{index=4}

# MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.3"
# MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

if torch.cuda.is_available():
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16,
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        device_map="auto",
        quantization_config=bnb_config,
        low_cpu_mem_usage=True,
    )
else:
    # CPU fallback (no bitsandbytes)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        device_map="auto",
        torch_dtype=torch.float32,
        low_cpu_mem_usage=True,
    )

model.eval()
print("✅ Loaded:", MODEL_ID)



tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

...

model-00002-of-00002.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/3.97G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

✅ Loaded: Qwen/Qwen2.5-3B-Instruct


In [ ]:
def llama_chat_generate(user_prompt, max_new_tokens=140, temperature=0.85, top_p=0.9):
    messages = [{"role": "user", "content": user_prompt}]
    input_ids = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            input_ids,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=top_p,
            repetition_penalty=1.05,
            eos_token_id=tokenizer.eos_token_id,
        )

    # Decode
    text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    # Lightweight extraction: keep last assistant turn-ish
    return text.split(messages[0]["content"])[-1].strip()


In [ ]:
print(llama_chat_generate("Write a short joke that includes the words 'spray' and 'chair'."))

assistant
Why did the chair go to the doctor? Because it had a lot of "backache," and the doctor suggested he get a back "spray."


In [ ]:
def generate_candidates(word1, word2, n=8):
    jokes = []
    base_prompt = (
        f"Write a short, original joke that includes the words "
        f"'{word1}' and '{word2}'."
    )
    for i in range(n):
        jokes.append(llama_chat_generate(base_prompt))
    return jokes


In [ ]:
cands = generate_candidates("spray", "chair", n=5)
for i, j in enumerate(cands, 1):
    print(f"{i}. {j}\n")


1. assistant
Why did the chair go to the doctor? Because it had a lot of aches and sprays!

2. assistant
Why did the chair go to the doctor? Because it had a lot of back issues! (Or perhaps, "Because it had a lot of spray issues!")

3. assistant
Why did the chair go to therapy? Because it had a lot of "back" problems, but now it sprays lavender oil to feel better!

4. assistant
Why did the chair go to the doctor? Because it had a lot of "backache," and the doctor suggested it try a new "spray."

5. assistant
Why did the chair go to the doctor? Because it had a lot of aches and “sprayed” them all over the room!



In [ ]:
HUMOR_STRATEGIES = [
    "use exaggeration",
    "use anthropomorphism",
    "use a surprising role reversal",
    "use a literal interpretation of a phrase",
    "use absurd cause-and-effect",
    "use self-deprecating humor",
]

def generate_candidates_with_strategy(word1, word2):
    jokes = []
    for strat in HUMOR_STRATEGIES:
        prompt = (
            f"Write a short, original joke that includes the words "
            f"'{word1}' and '{word2}'. "
            f"The joke should {strat}."
        )
        jokes.append(llama_chat_generate(prompt))
    return jokes


In [ ]:
jokes = generate_candidates_with_strategy("spray", "chair")
for j in jokes:
    print(j, "\n")


assistant
Why did the chair go to the doctor? Because it had a serious case of spilled coffee "spray" all over its backside! 

assistant
Why did the chair go to the doctor? Because it had a lot of "back" problems, but after the session, it sprayed some "pain relief" on its chair cushion! 

assistant
Why did the chair spray its socks? Because it had a dirty seat! 

assistant
Why did the chair go to the doctor? Because it had a lot of "backache," but after the scan, they found out it just needed a good spray! 

assistant
Why did the chair refuse to spray? Because it didn't want to get its legs wet! 

assistant
Why did the cloud decide to sit on a chair? Because it heard there would be a cloud spray coming! (self-deprecating part: And let’s hope they didn’t get sprayed themselves.) 



News Title

In [ ]:
import torch

def llama_chat_generate(prompt: str,
                        max_new_tokens: int = 40,
                        temperature: float = 0.7,
                        top_p: float = 0.9) -> str:
    # Make sure these exist in your notebook:
    # tokenizer = AutoTokenizer.from_pretrained(...)
    # model = AutoModelForCausalLM.from_pretrained(...)
    assert isinstance(prompt, str) and len(prompt) > 0

    # Ensure padding token exists
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=top_p,
            num_beams=1,
            pad_token_id=tokenizer.eos_token_id,
        )

    # Decode ONLY the newly generated part
    gen_ids = out[0][inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(gen_ids, skip_special_tokens=True).strip()

In [ ]:
import re
from typing import List, Dict

HEADLINE_STRATEGIES = [
    "satirical commentary (like late-night monologue)",
    "misdirection: set up serious, punchline unexpected",
    "analogy: compare the news to something everyday",
    "wordplay/pun (but keep it understandable)",
    "exaggeration to absurd extremes",
    "role reversal (objects/people switch roles)",
    "deadpan understatement",
    "mock 'product review' of the situation",
]

def clean_headline(h: str) -> str:
    h = (h or "").strip()
    h = re.sub(r"\s+", " ", h)
    return h

def headline_to_candidates(headline: str, k: int = 8) -> List[Dict]:
    headline = clean_headline(headline)
    out = []
    for strat in HEADLINE_STRATEGIES[:k]:
        prompt = f"""
You are a joke writer.
Write ONE short joke (1–2 sentences) inspired by this news headline:
"{headline}"

Constraints:
- Must clearly relate to the headline topic (use 1–2 key words from it).
- Be original; do not quote real articles.
- Avoid hate, harassment, slurs, explicit sexual content, or violence.
- End with a punchline.
Humor style: {strat}
"""
        joke = llama_chat_generate(prompt).strip()
        out.append({"strategy": strat, "joke": joke})
    return out

In [ ]:
headline = "Ryanair to cut 1 million more passenger seats ..."
cands = headline_to_candidates(headline, k=6)
for i, c in enumerate(cands, 1):
    print(i, c["strategy"])
    print(" ", c["joke"])
    print()

KeyboardInterrupt: 

KG

In [ ]:
import re
from collections import Counter, defaultdict

STOP = set("""
a an the and or but if then while to of in on at for with from by as is are was were be been being
this that these those it its it's i you we they he she them his her our your their
""".split())

def tokenize(headline: str):
    headline = (headline or "").lower()
    words = re.findall(r"[a-z']+", headline)
    words = [w for w in words if w not in STOP and len(w) > 2]
    return words

# Build: keyword -> Counter(other_keyword)
co = defaultdict(Counter)

def build_cooccurrence_kg(df_headlines, text_col="headline", min_df=3, top_per_word=80):
    # document frequency
    dfreq = Counter()
    docs = []
    for h in df_headlines[text_col].dropna().tolist():
        toks = sorted(set(tokenize(h)))
        docs.append(toks)
        dfreq.update(toks)

    # keep only reasonably frequent terms
    vocab = {w for w, c in dfreq.items() if c >= min_df}

    for toks in docs:
        toks = [w for w in toks if w in vocab]
        for i in range(len(toks)):
            for j in range(i+1, len(toks)):
                a, b = toks[i], toks[j]
                co[a][b] += 1
                co[b][a] += 1

    # trim neighbors
    co_trim = {}
    for w, ctr in co.items():
        co_trim[w] = ctr.most_common(top_per_word)
    return co_trim, dfreq

# Use your df_clean:
headline_df = df_clean[df_clean["headline"].notna()].copy()
co_kg, dfreq = build_cooccurrence_kg(headline_df, min_df=3)
print("KG nodes (keywords):", len(co_kg))

KG nodes (keywords): 837


In [ ]:
def kg_neighbors_for_headline(headline: str, co_kg, max_seeds=6, max_neighbors=12):
    seeds = [w for w in tokenize(headline) if w in co_kg]
    # prefer rarer-ish seeds (more interesting), but not super rare
    seeds = sorted(seeds, key=lambda w: dfreq[w])[:max_seeds]

    neigh = []
    seen = set(seeds)
    for s in seeds:
        for w, c in co_kg.get(s, [])[:50]:
            if w not in seen:
                neigh.append((w, c, s))
                seen.add(w)
            if len(neigh) >= max_neighbors:
                break
        if len(neigh) >= max_neighbors:
            break

    return seeds, neigh  # neigh items are (word, count, seed_that_led_to_it)

In [ ]:
def headline_to_candidates_kg(headline: str, k: int = 6):
    headline = clean_headline(headline)
    seeds, neigh = kg_neighbors_for_headline(headline, co_kg)

    twist_words = [w for (w, c, s) in neigh][:8]
    seeds_txt = ", ".join(seeds[:6]) if seeds else "(none)"
    twist_txt = ", ".join(twist_words) if twist_words else "(none)"

    out = []
    for strat in HEADLINE_STRATEGIES[:k]:
        prompt = f"""
Write ONE short joke (1–2 sentences) inspired by this news headline:
"{headline}"

KG hint (from similar headlines):
- headline keywords: {seeds_txt}
- related concepts to use for a twist: {twist_txt}

Constraints:
- Must clearly relate to the headline.
- Use 1–2 of the related concepts as the comedic twist (if safe/natural).
- Be original; do not quote real articles.
- Avoid hate, harassment, slurs, explicit sexual content, or violence.
- End with a punchline.
Humor style: {strat}
"""
        joke = llama_chat_generate(prompt).strip()
        out.append({"strategy": strat, "seeds": seeds, "twist_words": twist_words, "joke": joke})
    return out

In [ ]:
def headline_to_candidates_onecall(headline: str, k: int = 6):
    headline = clean_headline(headline)
    prompt = f"""
Write {k} different short jokes (1–2 sentences each) inspired by this headline:
"{headline}"

Constraints:
- Each joke must clearly relate to the headline.
- Each joke must use a different humor strategy (satire, misdirection, analogy, pun, exaggeration, deadpan).
- Be original; do not quote the article.
- Avoid hate, harassment, slurs, explicit sexual content, or violence.
Return as a numbered list 1..{k}.
"""
    text = llama_chat_generate(prompt, max_new_tokens=200)  # bigger because multiple jokes
    return text

In [ ]:
headline = "Ryanair to cut 1 million more passenger seats ..."
print(headline_to_candidates_onecall(headline, k=6))

1. In another move to squeeze every last penny from your wallet, Ryanair is planning to empty their pockets with even fewer seats. 
2. If you were hoping for a bigger window seat on your next flight, Ryanair has just announced they'll be shrinking all windows to match the new smaller seats.
3. Ryanair's latest plan is to cut so many seats, it might leave passengers wondering if there are any seats at all!
4. With this drastic measure, Ryanair is sending a clear message: your dreams of comfortable air travel have been canceled.
5. Imagine if you could only fit one arm on your seat and no legroom! That's the kind of cramped quarters Ryanair plans to offer.
6. The airline is now offering the ultimate in space efficiency: one person per seat, but no personal space. 

Note: While these jokes are creative and inspired by the headline, they should not be interpreted as endorsements of such practices. The aim here was to craft humorous


Test

In [ ]:
import pandas as pd

df = pd.read_csv("/content/task-a-en.tsv", sep="\t")
print(len(df))  # should be 300 for English


300


In [ ]:
def is_word_inclusion_row(row):
    return isinstance(row.get("word1"), str) and isinstance(row.get("word2"), str)


In [ ]:
def generate_word_joke(word1, word2, lang="en"):
    prompt = f"""
Write a short joke that includes BOTH of these words exactly as written:
"{word1}" and "{word2}"

Constraints:
- The joke must be written in {lang.upper()}.
- Be original and humorous.
- Avoid hate, harassment, slurs, explicit sexual content, or violence.
"""
    joke = llama_chat_generate(prompt, max_new_tokens=60)
    return joke.strip()


In [ ]:
def generate_headline_joke(headline, lang="en"):
    prompt = f"""
Write a short joke (1–2 sentences) inspired by this news headline:

"{headline}"

Constraints:
- The joke must be written in {lang.upper()}.
- It must clearly relate to the headline.
- Be original; do not quote the article.
- Avoid hate, harassment, slurs, explicit sexual content, or violence.
"""
    joke = llama_chat_generate(prompt, max_new_tokens=80)
    return joke.strip()


In [ ]:
def clip_length(text, lang):
    limits = {"en": 900, "es": 900, "zh": 300}
    return text[:limits[lang]]


In [ ]:
outputs = []

LANG = "en"  # change to "es" or "zh" per file

for _, row in df.iterrows():
    if is_word_inclusion_row(row):
        joke = generate_word_joke(row["word1"], row["word2"], LANG)
    else:
        joke = generate_headline_joke(row["headline"], LANG)

    joke = clip_length(joke, LANG)

    outputs.append({
        "id": row["id"],
        "text": joke
    })


KeyboardInterrupt: 

In [ ]:
!pip install -q transformers accelerate bitsandbytes pandas nltk spacy
!python -m spacy download en_core_web_sm -q
!python -m spacy download es_core_news_sm -q
!python -m spacy download zh_core_web_sm -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 10.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 138.9 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 92.3 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 MB 15.2 MB/s eta 0:00:00
   ━━

In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


In [ ]:
import nltk
nltk.download("wordnet")
nltk.download("omw-1.4")

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


True

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    quantization_config=bnb_config,
)
model.eval()
print("✅ Model loaded:", MODEL_ID)


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

✅ Model loaded: Qwen/Qwen2.5-1.5B-Instruct


In [ ]:
def chat_generate(prompt, max_new_tokens=80, temperature=0.7, top_p=0.9):
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=top_p,
            num_beams=1,
            pad_token_id=tokenizer.eos_token_id,
        )

    decoded = tokenizer.decode(out[0], skip_special_tokens=True)

    # Best-effort remove prompt echo
    if messages[0]["content"] in decoded:
        decoded = decoded.split(messages[0]["content"], 1)[-1]

    return decoded.strip()


In [ ]:
import pandas as pd, re, os, unicodedata
from nltk.corpus import wordnet as wn
import spacy

NLP = {
    "en": spacy.load("en_core_web_sm"),
    "es": spacy.load("es_core_news_sm"),
    "zh": spacy.load("zh_core_web_sm"),
}

LANG_CFG = {
    "en": {"max_chars": 900, "name": "English", "out": "task-a-en.tsv"},
    "es": {"max_chars": 900, "name": "Spanish", "out": "task-a-es.tsv"},
    "zh": {"max_chars": 300, "name": "Chinese", "out": "task-a-zh.tsv"},
}

def clean_cell(x):
    if pd.isna(x):
        return None
    x = str(x).strip()
    if x in ["-", "–", "—", ""]:
        return None
    return x

def normalize_text(text, max_chars):
    text = unicodedata.normalize("NFKC", str(text))
    text = re.sub(r"\s+", " ", text).strip()
    return text[:max_chars]

def wordnet_relations_en(word, maxn=3):
    # WordNet is English-focused; we still use it for “ideas”
    synsets = wn.synsets(word)
    rels = set()
    for s in synsets[:2]:
        for h in s.hypernyms():
            rels.add(f"{word} is a kind of {h.lemmas()[0].name().replace('_',' ')}")
        for l in s.lemmas():
            for a in l.antonyms():
                rels.add(f"{word} contrasts with {a.name().replace('_',' ')}")
    return list(rels)[:maxn]

def extract_entities(text, lang):
    doc = NLP[lang](text)
    ents = [ent.text for ent in getattr(doc, "ents", [])]
    # de-duplicate preserving order
    out = []
    for e in ents:
        if e not in out:
            out.append(e)
    return out[:6]


In [ ]:
def prompt_two_words(w1, w2, lang):
    if lang == "en":
        base = f'Write ONE short joke (1–2 sentences) that MUST include both words: "{w1}" and "{w2}".'
    elif lang == "es":
        base = f'Escribe UN chiste corto (1–2 frases) que DEBE incluir estas dos palabras: "{w1}" y "{w2}".'
    else:  # zh
        base = f'写一个简短的笑话（1–2句），必须同时包含这两个词：“{w1}” 和 “{w2}”。'

    kg = "; ".join(wordnet_relations_en(w1) + wordnet_relations_en(w2))

    if lang == "en":
        extra = f"\nOptional knowledge:\n{kg}\n\nNo offensive content. Keep it concise."
    elif lang == "es":
        extra = f"\nConocimiento opcional:\n{kg}\n\nSin contenido ofensivo. Sé conciso."
    else:
        extra = f"\n可选知识:\n{kg}\n\n避免冒犯内容。尽量简短。"

    return (base + extra).strip()

def prompt_headline(headline, lang):
    ents = ", ".join(extract_entities(headline, lang))
    if lang == "en":
        return f"""
Write ONE short joke (1–2 sentences) inspired by this news headline:
"{headline}"
Key concepts: {ents}
No offensive content. Keep it concise.
""".strip()

    if lang == "es":
        return f"""
Escribe UN chiste corto (1–2 frases) inspirado en este titular:
"{headline}"
Conceptos clave: {ents}
Sin contenido ofensivo. Sé conciso.
""".strip()

    return f"""
根据这个新闻标题写一个简短笑话（1–2句）：
“{headline}”
关键词：{ents}
避免冒犯内容。尽量简短。
""".strip()


In [ ]:
def generate_task_a(lang, input_file):
    cfg = LANG_CFG[lang]
    out_file = cfg["out"]
    max_chars = cfg["max_chars"]

    ckpt = f"{out_file}.partial"
    print(f"\n🚀 Generating {cfg['name']} from {input_file} → {out_file}")

    df = pd.read_csv(input_file, sep="\t")
    df = df.applymap(clean_cell)

    # Resume set
    done_ids = set()
    if os.path.exists(ckpt):
        tmp = pd.read_csv(ckpt, sep="\t")
        done_ids = set(tmp["id"].astype(str).tolist())
        print(f"↩️ Resuming: {len(done_ids)} already done")

    buffer = []
    def flush():
        nonlocal buffer
        if not buffer:
            return
        mode = "a" if os.path.exists(ckpt) else "w"
        header = not os.path.exists(ckpt)
        pd.DataFrame(buffer).to_csv(ckpt, sep="\t", index=False, mode=mode, header=header)
        buffer = []
        torch.cuda.empty_cache()

    for i, row in df.iterrows():
        _id = str(row["id"])
        if _id in done_ids:
            continue

        w1, w2 = row.get("word1"), row.get("word2")
        headline = row.get("headline")

        if w1 and w2:
            prompt = prompt_two_words(w1, w2, lang)
            text = chat_generate(prompt, max_new_tokens=80)
        else:
            prompt = prompt_headline(headline, lang)
            text = chat_generate(prompt, max_new_tokens=90)

        text = normalize_text(text, max_chars)
        buffer.append({"id": _id, "text": text})

        if len(buffer) >= 10:
            flush()

        if (i + 1) % 25 == 0:
            print(f"{cfg['name']} progress: {i+1}/{len(df)}")

    flush()

    # Build final output in correct order
    partial = pd.read_csv(ckpt, sep="\t")
    merged = df[["id"]].merge(partial, on="id", how="left")

    # Fallback if any missing (should be none if completed)
    fallback = {
        "en": "I tried to write a joke, but the punchline missed its connection.",
        "es": "Intenté escribir un chiste, pero el remate perdió la conexión.",
        "zh": "我想写个笑话，但笑点没赶上。"
    }[lang]
    merged["text"] = merged["text"].fillna(fallback).map(lambda x: normalize_text(x, max_chars))

    merged.to_csv(out_file, sep="\t", index=False)
    print(f"✅ Wrote {out_file} rows={len(merged)}")


In [ ]:
generate_task_a("en", "task-a-en.tsv")


🚀 Generating English from task-a-en.tsv → task-a-en.tsv


/tmp/ipython-input-3834418138.py:10: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(clean_cell)


English progress: 25/300
English progress: 50/300
English progress: 75/300
English progress: 100/300
English progress: 125/300
English progress: 150/300
English progress: 175/300
English progress: 200/300
English progress: 225/300
English progress: 250/300
English progress: 275/300
English progress: 300/300
✅ Wrote task-a-en.tsv rows=300


In [ ]:
generate_task_a("es", "task-a-es.tsv")


🚀 Generating Spanish from task-a-es.tsv → task-a-es.tsv


/tmp/ipython-input-3834418138.py:10: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(clean_cell)


Spanish progress: 25/300
Spanish progress: 50/300
Spanish progress: 75/300
Spanish progress: 100/300
Spanish progress: 125/300
Spanish progress: 150/300
Spanish progress: 175/300
Spanish progress: 200/300
Spanish progress: 225/300
Spanish progress: 250/300
Spanish progress: 275/300
Spanish progress: 300/300
✅ Wrote task-a-es.tsv rows=300


In [ ]:
generate_task_a("zh", "task-a-zh.tsv")


🚀 Generating Chinese from task-a-zh.tsv → task-a-zh.tsv


/tmp/ipython-input-3834418138.py:10: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(clean_cell)


Chinese progress: 25/300
Chinese progress: 50/300
Chinese progress: 75/300
Chinese progress: 100/300
Chinese progress: 125/300
Chinese progress: 150/300
Chinese progress: 175/300
Chinese progress: 200/300
Chinese progress: 225/300
Chinese progress: 250/300
Chinese progress: 275/300
Chinese progress: 300/300
✅ Wrote task-a-zh.tsv rows=300


In [ ]:
from google.colab import files
files.download("task-a-en.tsv")
files.download("task-a-es.tsv")
files.download("task-a-zh.tsv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>